# General Notebook

This notebook uses a custom kernel with all required dependencies installed.


In [1]:
# Check installed packages
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")


Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]
Python executable: c:\Users\Hossein\.conda\envs\general_env\python.exe


In [2]:
# Import common libraries
try:
    import numpy as np
    import boto3
    import requests
    from tqdm import tqdm
    import zarr
    print("All dependencies imported successfully!")
except ImportError as e:
    print(f"Missing dependency: {e}")


All dependencies imported successfully!


In [3]:
# Read biomedvis-6gb data and print metadata
import zarr
from pathlib import Path
data_path = Path("Data/biomedvis-6gb/0/3")
print(f"{'='*60}\nDataset: biomedvis-6gb\n{'='*60}")
try:
    zarr_array = zarr.open(str(data_path), mode='r')
    print(f"\nPath: {data_path}")
    print(f"Type: Zarr Array")
    print(f"Shape: {zarr_array.shape}")
    print(f"Dtype: {zarr_array.dtype}")
    print(f"Chunks: {zarr_array.chunks}")
    # Print metadata from attributes
    if hasattr(zarr_array, 'attrs') and zarr_array.attrs:
        attrs = zarr_array.attrs
        print(f"\nAttributes keys: {list(attrs.keys())}")
        if 'multiscales' in attrs:
            scales = attrs['multiscales'][0]
            print(f"\n  Name: {scales.get('name', 'N/A')}")
            print(f"  Axes: {[ax['name'] for ax in scales.get('axes', [])]}")
            print(f"  Resolution levels: {len(scales.get('datasets', []))}")
        if 'omero' in attrs:
            channels = attrs['omero'].get('channels', [])
            channel_names = [c['label'] for c in channels]
            print(f"  All channels ({len(channel_names)}): {channel_names}")
except Exception as e:
    print(f"Error reading zarr data: {e}")

Dataset: biomedvis-6gb

Path: Data\biomedvis-6gb\0\3
Type: Zarr Array
Shape: (1, 70, 194, 688, 1363)
Dtype: >u2
Chunks: (1, 1, 1, 688, 1024)


In [4]:
# Load data as dask array
import dask.array as da
import zarr
from pathlib import Path
from IPython.display import HTML, display

# Path to zarr array (component 3)
path = Path("Data/biomedvis-6gb/0/3")

# Check if path exists
if not path.exists():
    raise FileNotFoundError(f"Path not found: {path.absolute()}. Please check the data directory structure.")

# Load directly from zarr array path
daskArray = da.from_zarr(str(path))

# Display detailed information in Colab-like format
def format_bytes(bytes_size):
    """Format bytes to human readable format"""
    for unit in ['B', 'KiB', 'MiB', 'GiB', 'TiB']:
        if bytes_size < 1024.0:
            return f"{bytes_size:.2f} {unit}"
        bytes_size /= 1024.0
    return f"{bytes_size:.2f} PiB"

# Calculate number of chunks
num_chunks = daskArray.numblocks
total_chunks = 1
for n in num_chunks:
    total_chunks *= n

# Calculate chunk size in bytes
chunk_size_bytes = 1
for cs in daskArray.chunksize:
    chunk_size_bytes *= cs
chunk_size_bytes *= daskArray.dtype.itemsize

# Display information in Colab-like format
print("Array\tChunk")
print(f"Bytes\t{format_bytes(daskArray.nbytes)}\t{format_bytes(chunk_size_bytes)}")
print(f"Shape\t{daskArray.shape}\t{daskArray.chunksize}")
print(f"Dask graph\t{total_chunks} chunks in 2 graph layers")
print(f"Data type\t{daskArray.dtype} numpy.ndarray")
print()

Array	Chunk
Bytes	23.72 GiB	1.34 MiB
Shape	(1, 70, 194, 688, 1363)	(1, 1, 1, 688, 1024)
Dask graph	27160 chunks in 2 graph layers
Data type	>u2 numpy.ndarray



In [5]:
# Reading the Physical Size of the Sample
from pathlib import Path
import zarr

# Path relative to notebook location (component 3)
path = Path("Data/biomedvis-6gb/0/3")

# Read zarr array and extract physical size from multiscales metadata
zarr_array = zarr.open(str(path), mode='r')
attrs = zarr_array.attrs

if 'multiscales' in attrs:
    scales = attrs['multiscales'][0]
    datasets = scales.get('datasets', [])
    if datasets:
        transforms = datasets[0].get('coordinateTransformations', [])
        if transforms:
            scale = transforms[0].get('scale', [])  # [t, c, z, y, x] in micrometers
            if len(scale) >= 5:
                physical_size_x = scale[4]  # x dimension
                physical_size_y = scale[3]  # y dimension  
                physical_size_z = scale[2]  # z dimension
                print(f"Physical Size X: {physical_size_x} μm")
                print(f"Physical Size Y: {physical_size_y} μm")
                print(f"Physical Size Z: {physical_size_z} μm")
                print(f"\nPhysical Size: ({physical_size_x}, {physical_size_y}, {physical_size_z}) μm")
                # Store for later use
                ome_xml_physical_size = (physical_size_x, physical_size_y, physical_size_z)


In [6]:
# Getting the raw data from Channel CD31
# CD31 is at channel index 19 (0-indexed)
# daskArray shape: (t, c, z, y, x) = (1, 70, 194, 688, 1363)

# Get data for t=0, channel=19 (CD31), all z, y, x
# Uses daskArray from previous cell (Cell 4)
data = daskArray[0, 19, :, :, :].compute()

print(f"Channel: CD31 (index 19)")
print(f"Data shape: {data.shape}")
print(f"Data dtype: {data.dtype}")
print(f"Data range: [{data.min()}, {data.max()}]")
print(f"Data mean: {data.mean():.2f}")
print(f"Data size: {data.nbytes / (1024**2):.2f} MB")

data.shape


Channel: CD31 (index 19)
Data shape: (194, 688, 1363)
Data dtype: >u2
Data range: [0, 32737]
Data mean: 13.17
Data size: 346.99 MB


(194, 688, 1363)

## 3D Visualization Options

**Best Methods for Large Volume Data:**

1. **Napari** (Recommended) - Interactive volume visualization, supports large datasets with lazy loading
2. **Plotly** - WebGL-based 3D visualization, good for interactive notebooks
3. **Three.js/WebGL** - Custom HTML viewer with volume rendering
4. **PyVista** - 3D scientific visualization

**Note:** Due to large size (194 x 688 x 1363), we'll use downsampling or volume rendering techniques.


In [7]:
# Method 1: Napari - Best for interactive volume visualization
# Install: pip install napari[all]

try:
    import napari
    from skimage import exposure
    
    # Select channel (channel index 0 = first channel)
    channel_idx = 19
    print(f"Loading channel {channel_idx} for 3D visualization...")

    
    
    # Load channel data (downsample if needed for better performance)
    # Option 1: Use full resolution (may be slow)
    # channel_data = daskArray[0, channel_idx, :, :, :].compute()
    
    # Option 2: Downsample for better performance (recommended)
    # Downsample by factor of 2 in each dimension
    channel_data = daskArray[0, channel_idx, ::1, ::1, ::1].compute()
    print(f"Data shape (downsampled): {channel_data.shape}")
    print(f"Data range: [{channel_data.min()}, {channel_data.max()}]")
    
    # Normalize data for visualization (0-255 range)
    channel_data_normalized = exposure.rescale_intensity(
        channel_data, 
        in_range=(channel_data.min(), channel_data.max()),
        out_range=(0, 255)
    ).astype('uint8')
    
    # Create napari viewer
    viewer = napari.Viewer()
    viewer.add_image(
        channel_data_normalized,
        name=f'Channel {channel_idx}',
        rendering='iso',  # Iso-surface rendering
        opacity=0.7,
        contrast_limits=[0, 255]
    )
    print("\nNapari viewer opened. Use mouse to rotate/zoom the 3D volume.")
    print("Controls:")
    print("  - Left mouse: Rotate")
    print("  - Right mouse: Pan")
    print("  - Scroll: Zoom")
    print("  - 'Iso' button: Toggle iso-surface rendering")
    
except ImportError:
    print("Napari not installed. Install with: pip install napari[all]")
    print("Or try Method 2 (Plotly) below.")
except Exception as e:
    print(f"Error with Napari: {e}")
    print("Trying alternative method...")


Loading channel 19 for 3D visualization...
Data shape (downsampled): (194, 688, 1363)
Data range: [0, 32737]

Napari viewer opened. Use mouse to rotate/zoom the 3D volume.
Controls:
  - Left mouse: Rotate
  - Right mouse: Pan
  - Scroll: Zoom
  - 'Iso' button: Toggle iso-surface rendering


In [8]:
# Method 2: Plotly - WebGL 3D Visualization
import plotly.graph_objects as go
import numpy as np
from plotly.offline import plot
from pathlib import Path
from IPython.display import HTML, display

channel_idx = 0

# Load and downsample data
channel_data = daskArray[0, channel_idx, ::8, ::2, ::2].compute()
print(f"Channel {channel_idx} shape: {channel_data.shape}")

# Normalize to 0-255
channel_data_norm = ((channel_data - channel_data.min()) / 
                     (channel_data.max() - channel_data.min()) * 255).astype(np.uint8)

# Create point cloud (sample every 2nd point)
sample_step = 2
z_coords, y_coords, x_coords = np.meshgrid(
    np.arange(channel_data_norm.shape[0])[::sample_step],
    np.arange(channel_data_norm.shape[1])[::sample_step],
    np.arange(channel_data_norm.shape[2])[::sample_step],
    indexing='ij'
)
sampled_data = channel_data_norm[::sample_step, ::sample_step, ::sample_step]

# Filter points with intensity > 10th percentile
values = sampled_data.flatten()
threshold = np.percentile(values, 10)
mask = values > threshold

# Create visualization
fig = go.Figure(data=go.Scatter3d(
    x=x_coords.flatten()[mask],
    y=y_coords.flatten()[mask],
    z=z_coords.flatten()[mask],
    mode='markers',
    marker=dict(
        size=2,
        color=values[mask],
        colorscale='Viridis',
        opacity=0.6,
        showscale=True
    )
))

fig.update_layout(
    title=f'3D Visualization - Channel {channel_idx}',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    width=1200,
    height=900
)

# Save and display
output_dir = Path("visualization_data")
output_dir.mkdir(exist_ok=True)
html_file = output_dir / f"plotly_3d_channel_{channel_idx}.html"
plot(fig, filename=str(html_file), auto_open=False)

print(f"✓ Saved to: {html_file}")
try:
    display(HTML(plot(fig, output_type='div', include_plotlyjs='cdn')))
except:
    pass


Channel 0 shape: (25, 344, 682)
✓ Saved to: visualization_data\plotly_3d_channel_0.html


In [9]:
# Method 3: Prepare data for Three.js WebGL viewer (export to format usable by web)
# This creates a compressed representation that can be loaded in a Three.js viewer

import json
import numpy as np
from pathlib import Path

def prepare_data_for_threejs(channel_idx=0, downsample_factor=1):
    """
    Prepare channel data for Three.js volume rendering.
    Returns data in a format suitable for web transfer.
    """
    print(f"Preparing channel {channel_idx} data for Three.js...")
    
    # Load and downsample
    channel_data = daskArray[0, channel_idx, ::downsample_factor, ::downsample_factor, ::downsample_factor].compute()
    print(f"Shape: {channel_data.shape}")
    
    # Normalize to 0-255
    data_min, data_max = channel_data.min(), channel_data.max()
    channel_data_norm = ((channel_data - data_min) / (data_max - data_min) * 255).astype(np.uint8)
    
    # Save metadata
    metadata = {
        'shape': channel_data.shape,
        'dataRange': [int(data_min), int(data_max)],
        'downsampleFactor': downsample_factor,
        'channel': channel_idx
    }
    
    # Option 1: Save as compressed numpy array (can be loaded in JavaScript with pako.js)
    output_dir = Path("visualization_data")
    output_dir.mkdir(exist_ok=True)
    
    # Save metadata
    with open(output_dir / f"channel_{channel_idx}_metadata.json", 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Save data as binary (can be loaded with fetch in JS)
    channel_data_norm.tofile(output_dir / f"channel_{channel_idx}_data.raw")
    
    print(f"\nData prepared for Three.js viewer!")
    print(f"Files saved to: visualization_data")
    print(f"  - channel_{channel_idx}_metadata.json")
    print(f"  - channel_{channel_idx}_data.raw")
    print(f"\nYou can now create an HTML file with Three.js to load and render this data.")
    print(f"Data size: {channel_data_norm.nbytes / (1024**2):.2f} MB")
    
    return metadata, channel_data_norm

# Prepare data for channel 0
metadata, prepared_data = prepare_data_for_threejs(channel_idx=0, downsample_factor=1)


Preparing channel 0 data for Three.js...
Shape: (194, 688, 1363)

Data prepared for Three.js viewer!
Files saved to: visualization_data
  - channel_0_metadata.json
  - channel_0_data.raw

You can now create an HTML file with Three.js to load and render this data.
Data size: 173.49 MB


In [10]:
# Method 4: Three.js WebGL Viewer - Same data as Napari (Channel 19/CD31)
# This creates an interactive Three.js HTML viewer for the same channel shown in Napari

import json
import numpy as np
from pathlib import Path
from skimage import exposure
from IPython.display import HTML, display

# Use the same channel and settings as Napari (Cell 8)
channel_idx = 19  # CD31 channel
print(f"Preparing channel {channel_idx} (CD31) for Three.js visualization...")
print("This is the same data shown in Napari viewer (Cell 8).")

# Load channel data with same settings as Napari
channel_data = daskArray[0, channel_idx, ::1, ::1, ::1].compute()
print(f"Data shape: {channel_data.shape}")
print(f"Data range: [{channel_data.min()}, {channel_data.max()}]")

# Normalize data for visualization (same as Napari)
# Use manual normalization to avoid MemoryError with large arrays
# Calculate min/max first
data_min = float(channel_data.min())
data_max = float(channel_data.max())
print(f"Data min: {data_min}, max: {data_max}")

# Normalize in-place to avoid creating intermediate float64 arrays
# Use float32 instead of float64 to save memory
if data_max > data_min:
    # Normalize to 0-255 range using float32 for intermediate calculations
    channel_data_normalized = ((channel_data.astype('float32') - data_min) / (data_max - data_min) * 255).astype('uint8')
else:
    # If all values are the same, set to 128
    channel_data_normalized = np.full_like(channel_data, 128, dtype='uint8')

print("Using full resolution (no downsampling)")

# Prepare metadata
metadata = {
    'shape': list(channel_data_normalized.shape),
    'dataRange': [int(data_min), int(data_max)],
    'channel': channel_idx,
    'channelName': 'CD31'
}

# Save data and metadata
output_dir = Path("visualization_data")
output_dir.mkdir(exist_ok=True)

# Save metadata
metadata_file = output_dir / f"channel_{channel_idx}_napari_metadata.json"
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

# Save data as binary
data_file = output_dir / f"channel_{channel_idx}_napari_data.raw"
channel_data_normalized.tofile(data_file)

print(f"\n✓ Data prepared for Three.js viewer!")
print(f"✓ Files saved to: {output_dir}")
print(f"  - {metadata_file.name}")
print(f"  - {data_file.name}")
print(f"✓ Data size: {channel_data_normalized.nbytes / (1024**2):.2f} MB")

# Create Three.js HTML viewer
html_content = f'''<!DOCTYPE html>
<html>
<head>
    <title>3D Volume Viewer - Channel {channel_idx} (CD31) - Three.js</title>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
    <style>
        body {{
            margin: 0;
            overflow: hidden;
            font-family: Arial, sans-serif;
            background: #000;
        }}
        #info {{
            position: absolute;
            top: 10px;
            left: 10px;
            color: white;
            background: rgba(0,0,0,0.8);
            padding: 15px;
            border-radius: 5px;
            z-index: 100;
            max-width: 300px;
        }}
        #controls {{
            position: absolute;
            top: 10px;
            right: 10px;
            color: white;
            background: rgba(0,0,0,0.8);
            padding: 15px;
            border-radius: 5px;
            min-width: 200px;
            z-index: 100;
        }}
        .control-group {{
            margin: 10px 0;
        }}
        label {{
            display: block;
            margin-bottom: 5px;
            font-size: 12px;
        }}
        input[type="range"] {{
            width: 100%;
        }}
        button {{
            padding: 8px 15px;
            margin: 5px 0;
            cursor: pointer;
            background: #4CAF50;
            color: white;
            border: none;
            border-radius: 3px;
            width: 100%;
        }}
        button:hover {{
            background: #45a049;
        }}
        #loading {{
            position: absolute;
            top: 50%;
            left: 50%;
            transform: translate(-50%, -50%);
            color: white;
            font-size: 18px;
            z-index: 200;
        }}
    </style>
</head>
<body>
    <div id="loading">Loading volume data...</div>
    <div id="info">
        <h3>3D Volume Viewer</h3>
        <p><strong>Channel:</strong> {channel_idx} (CD31)</p>
        <p><strong>Dimensions:</strong> Z={metadata['shape'][0]}, Y={metadata['shape'][1]}, X={metadata['shape'][2]}</p>
        <p><strong>Physical Size:</strong> Z=0.28 μm, Y=0.14 μm, X=0.14 μm</p>
        <p><strong>Resolution:</strong> Full (no downsampling)</p>
        <p><strong>Aspect Ratio:</strong> Z compressed 4x to match physical proportions</p>
        <p><strong>Controls:</strong></p>
        <ul style="font-size: 11px; margin: 5px 0;">
            <li>Left Click + Drag: Rotate</li>
            <li>Right Click + Drag: Pan</li>
            <li>Scroll: Zoom</li>
        </ul>
    </div>
    
    <div id="controls">
        <h3 style="margin-top: 0;">Settings</h3>
        <div class="control-group">
            <label>Threshold: <span id="thresholdValue">10</span></label>
            <input type="range" id="threshold" min="0" max="255" value="10" oninput="updateThreshold(this.value)">
        </div>
        <div class="control-group">
            <label>Sphere Radius: <span id="sizeValue">0.006</span></label>
            <input type="range" id="pointSize" min="1" max="10" value="5" oninput="updatePointSize(this.value)">
        </div>
        <div class="control-group">
            <label>Opacity: <span id="opacityValue">0.6</span></label>
            <input type="range" id="opacity" min="0" max="100" value="60" oninput="updateOpacity(this.value)">
        </div>
        <div class="control-group">
            <label>Downsampling: <span id="samplingValue">1</span></label>
            <select id="sampling" onchange="updateSampling(this.value)" style="width: 100%; padding: 5px;">
                <option value="1" selected>1 (No downsampling)</option>
                <option value="2">2 (2x downsampling)</option>
                <option value="4">4 (4x downsampling)</option>
            </select>
        </div>
        <button onclick="resetCamera()">Reset Camera</button>
    </div>
    
    <script>
        const metadata = {json.dumps(metadata, separators=(',', ':'))};
        const dataFile = 'channel_{channel_idx}_napari_data.raw';
        let scene, camera, renderer, pointCloud, cameraLight, cameraPointLight;
        
        // Scene setup
        scene = new THREE.Scene();
        camera = new THREE.PerspectiveCamera(75, window.innerWidth / window.innerHeight, 0.1, 1000);
        renderer = new THREE.WebGLRenderer({{ antialias: true }});
        renderer.setSize(window.innerWidth, window.innerHeight);
        renderer.setClearColor(0x000000);
        document.body.appendChild(renderer.domElement);
        
        // Add lights - improved lighting to avoid dark spots
        // Stronger ambient light for overall illumination
        const ambientLight = new THREE.AmbientLight(0xffffff, 1.0);
        scene.add(ambientLight);
        
        // Multiple directional lights from different angles for better coverage
        const directionalLight1 = new THREE.DirectionalLight(0xffffff, 0.9);
        directionalLight1.position.set(1, 1, 1);
        scene.add(directionalLight1);
        
        const directionalLight2 = new THREE.DirectionalLight(0xffffff, 0.7);
        directionalLight2.position.set(-1, 1, -1);
        scene.add(directionalLight2);
        
        const directionalLight3 = new THREE.DirectionalLight(0xffffff, 0.6);
        directionalLight3.position.set(1, -1, 1);
        scene.add(directionalLight3);
        
        const directionalLight4 = new THREE.DirectionalLight(0xffffff, 0.5);
        directionalLight4.position.set(-1, -1, -1);
        scene.add(directionalLight4);
        
        // Hemisphere light for natural sky/ground lighting
        const hemisphereLight = new THREE.HemisphereLight(0xffffff, 0x444444, 0.8);
        scene.add(hemisphereLight);
        
        // Camera-attached directional light that points from camera to data center
        // This light always points from camera position toward the origin (data center)
        cameraLight = new THREE.DirectionalLight(0xffffff, 3.0);
        cameraLight.target.position.set(0, 0, 0); // Point toward data center
        scene.add(cameraLight);
        scene.add(cameraLight.target); // Need to add target to scene for directional light
        
        // Additional point light at camera position pointing toward data
        cameraPointLight = new THREE.PointLight(0xffffff, 2.5, 100);
        scene.add(cameraPointLight);
        
        // Camera controls
        let isRotating = false;
        let isPanning = false;
        let mouseX = 0, mouseY = 0;
        let cameraRotation = {{ x: 0.5, y: 0.5 }};
        let cameraDistance = 3;
        let panOffset = {{ x: 0, y: 0 }};
        
        renderer.domElement.addEventListener('mousedown', (e) => {{
            if (e.button === 0) isRotating = true;
            if (e.button === 2) isPanning = true;
            mouseX = e.clientX;
            mouseY = e.clientY;
        }});
        
        renderer.domElement.addEventListener('mouseup', () => {{
            isRotating = false;
            isPanning = false;
        }});
        
        renderer.domElement.addEventListener('contextmenu', (e) => e.preventDefault());
        
        renderer.domElement.addEventListener('mousemove', (e) => {{
            if (isRotating) {{
                cameraRotation.y += (e.clientX - mouseX) * 0.01;
                cameraRotation.x += (e.clientY - mouseY) * 0.01;
                updateCameraPosition();
            }}
            if (isPanning) {{
                panOffset.x += (e.clientX - mouseX) * 0.001;
                panOffset.y -= (e.clientY - mouseY) * 0.001;
                updateCameraPosition();
            }}
            mouseX = e.clientX;
            mouseY = e.clientY;
        }});
        
        renderer.domElement.addEventListener('wheel', (e) => {{
            cameraDistance *= (1 + e.deltaY * 0.001);
            cameraDistance = Math.max(0.5, Math.min(10, cameraDistance));
            updateCameraPosition();
        }});
        
        function updateCameraPosition() {{
            const radius = cameraDistance;
            camera.position.x = radius * Math.sin(cameraRotation.y) * Math.cos(cameraRotation.x) + panOffset.x;
            camera.position.y = radius * Math.sin(cameraRotation.x) + panOffset.y;
            camera.position.z = radius * Math.cos(cameraRotation.y) * Math.cos(cameraRotation.x);
            camera.lookAt(panOffset.x, panOffset.y, 0);
            
            // Update camera light to point from camera position toward data center (origin)
            if (cameraLight) {{
                // Directional light: position at camera, pointing toward data center
                // For directional light, the light direction is from position to target
                cameraLight.position.copy(camera.position);
                // Target points toward data center (origin with pan offset)
                cameraLight.target.position.set(panOffset.x, panOffset.y, 0);
            }}
            
            // Update point light position to follow camera
            if (cameraPointLight) {{
                cameraPointLight.position.copy(camera.position);
            }}
        }}
        
        updateCameraPosition();
        
        // Load and visualize volume data
        fetch(dataFile)
            .then(response => response.arrayBuffer())
            .then(buffer => {{
                const data = new Uint8Array(buffer);
                document.getElementById('loading').style.display = 'none';
                createVolumeVisualization(data);
            }})
            .catch(error => {{
                console.error('Error loading data:', error);
                document.getElementById('loading').innerHTML = '<p style="color:red;">Error loading data file. Make sure the data file exists.</p>';
            }});
        
        let sphereGeometry = null; // Store sphere geometry reference for cleanup
        
        function createVolumeVisualization(data) {{
            const shape = metadata.shape;
            const [zSize, ySize, xSize] = shape;
            
            // Clear existing point cloud
            if (pointCloud) {{
                scene.remove(pointCloud);
                if (pointCloud.geometry) pointCloud.geometry.dispose();
                if (pointCloud.material) pointCloud.material.dispose();
                if (pointCloud.dispose) pointCloud.dispose();
            }}
            // Clean up previous sphere geometry
            if (sphereGeometry) {{
                sphereGeometry.dispose();
                sphereGeometry = null;
            }}
            
            // Get settings
            const threshold = parseInt(document.getElementById('threshold').value);
            // Convert slider value (1-10) to sphere radius (0.002-0.01)
            const sliderValue = parseFloat(document.getElementById('pointSize').value);
            const pointSize = 0.002 + (sliderValue - 1) * (0.01 - 0.002) / 9;
            const opacity = parseFloat(document.getElementById('opacity').value) / 100;
            const sampling = parseInt(document.getElementById('sampling').value);
            
            // Create point cloud
            const points = [];
            const colors = [];
            
            // Sample and filter points
            for (let z = 0; z < zSize; z += sampling) {{
                for (let y = 0; y < ySize; y += sampling) {{
                    for (let x = 0; x < xSize; x += sampling) {{
                        const idx = z * ySize * xSize + y * xSize + x;
                        const value = data[idx];
                        
                        if (value > threshold) {{
                            // Normalize coordinates with correct aspect ratio
                            // Physical size: z=0.28 μm, x=0.14 μm, y=0.14 μm
                            // Physical ratio: z/x = 0.28/0.14 = 2, z/y = 0.28/0.14 = 2
                            // So z should be compressed 4 times (not 8) to match physical proportions
                            const maxDim = Math.max(zSize, ySize, xSize);
                            const scaleX = xSize / maxDim;
                            const scaleY = ySize / maxDim;
                            const scaleZ = (zSize / maxDim) / 4;  // Divide z by 4 to match physical aspect ratio
                            
                            // Normalize to [-1, 1] with aspect ratio
                            const nx = ((x / xSize) * 2 - 1) * scaleX;
                            const ny = ((y / ySize) * 2 - 1) * scaleY;
                            const nz = ((z / zSize) * 2 - 1) * scaleZ;
                            
                            points.push(nx, ny, nz);
                            
                            // Fixed yellow color with intensity based on normalized value (0-255)
                            // Normalized data (0-255) maps to color intensity (0-1)
                            // Yellow color: RGB(255, 255, 0) = (1, 1, 0)
                            // Intensity: normalized value (0-255) maps to brightness (0-1)
                            const intensity = value / 255; // Normalized intensity 0-1 from normalized data (0-255)
                            // Yellow color with intensity: R=1*intensity, G=1*intensity, B=0
                            const r = intensity; // Yellow red component scaled by intensity
                            const g = intensity; // Yellow green component scaled by intensity
                            const b = 0; // No blue in yellow
                            
                            colors.push(r, g, b);
                        }}
                    }}
                }}
            }}
            
            // Create sphere geometry for each point
            const numPoints = points.length / 3;
            sphereGeometry = new THREE.SphereGeometry(pointSize, 8, 8); // radius, widthSegments, heightSegments
            
            // Create instanced mesh for better performance with many spheres
            const instancedMesh = new THREE.InstancedMesh(sphereGeometry, null, numPoints);
            
            // Create material with vertex colors support
            // Using MeshLambertMaterial for better color visibility and shiny appearance
            // Yellow emissive for better color visibility
            const material = new THREE.MeshLambertMaterial({{
                vertexColors: false,
                transparent: true,
                opacity: opacity,
                emissive: 0xffff00,  // Yellow emissive color
                emissiveIntensity: 0.4  // Higher self-illumination to show yellow color better
            }});
            
            // Set up instances
            const matrix = new THREE.Matrix4();
            const color = new THREE.Color();
            
            for (let i = 0; i < numPoints; i++) {{
                // Set position
                matrix.makeTranslation(points[i * 3], points[i * 3 + 1], points[i * 3 + 2]);
                instancedMesh.setMatrixAt(i, matrix);
                
                // Set color: fixed yellow with intensity based on normalized value (0-255)
                // Color is fixed yellow (RGB: 1, 1, 0), intensity shows the normalized value (0-1)
                // Normalized data (0-255) maps to color intensity (0-1)
                color.setRGB(colors[i * 3], colors[i * 3 + 1], colors[i * 3 + 2]);
                instancedMesh.setColorAt(i, color);
            }}
            
            // Enable color attribute
            instancedMesh.instanceColor.needsUpdate = true;
            instancedMesh.material = material;
            
            // Clear existing point cloud
            if (pointCloud) {{
                scene.remove(pointCloud);
                if (pointCloud.geometry) pointCloud.geometry.dispose();
                if (pointCloud.material) pointCloud.material.dispose();
            }}
            
            pointCloud = instancedMesh;
            scene.add(pointCloud);
            
            console.log(`Created ${{numPoints}} spheres`);
        }}
        
        function updateThreshold(value) {{
            document.getElementById('thresholdValue').textContent = value;
            fetch(dataFile)
                .then(response => response.arrayBuffer())
                .then(buffer => {{
                    const data = new Uint8Array(buffer);
                    createVolumeVisualization(data);
                }});
        }}
        
        function updatePointSize(value) {{
            // Convert slider value (1-10) to sphere radius (0.001-0.01)
            const sliderValue = parseFloat(value);
            const pointSize = 0.001 + (sliderValue - 1) * (0.01 - 0.001) / 9;
            document.getElementById('sizeValue').textContent = pointSize.toFixed(3);
            // Need to recreate visualization with new sphere size
            if (pointCloud) {{
                fetch(dataFile)
                    .then(response => response.arrayBuffer())
                    .then(buffer => {{
                        const data = new Uint8Array(buffer);
                        createVolumeVisualization(data);
                    }});
            }}
        }}
        
        function updateOpacity(value) {{
            const opacity = parseFloat(value) / 100;
            document.getElementById('opacityValue').textContent = opacity.toFixed(2);
            if (pointCloud && pointCloud.material) {{
                pointCloud.material.opacity = opacity;
            }}
        }}
        
        function updateSampling(value) {{
            document.getElementById('samplingValue').textContent = value;
            fetch(dataFile)
                .then(response => response.arrayBuffer())
                .then(buffer => {{
                    const data = new Uint8Array(buffer);
                    createVolumeVisualization(data);
                }});
        }}
        
        function resetCamera() {{
            cameraRotation = {{ x: 0.5, y: 0.5 }};
            cameraDistance = 3;
            panOffset = {{ x: 0, y: 0 }};
            updateCameraPosition();
        }}
        
        // Animation loop
        function animate() {{
            requestAnimationFrame(animate);
            renderer.render(scene, camera);
        }}
        animate();
        
        // Handle window resize
        window.addEventListener('resize', () => {{
            camera.aspect = window.innerWidth / window.innerHeight;
            camera.updateProjectionMatrix();
            renderer.setSize(window.innerWidth, window.innerHeight);
        }});
    </script>
</body>
</html>'''

# Save HTML file
html_file = output_dir / f"threejs_volume_viewer_channel_{channel_idx}_napari.html"
with open(html_file, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\n✓ Three.js HTML viewer created: {html_file}")
print(f"✓ Open this file in your browser to view the 3D volume")
print(f"✓ This displays the same data as shown in Napari (Cell 8)")

# Try to display in notebook
try:
    from IPython.display import IFrame
    display(IFrame(str(html_file), width=1200, height=800))
except Exception as e:
    print(f"\nNote: To view in notebook, open the HTML file directly:")
    print(f"  {html_file.absolute()}")


Preparing channel 19 (CD31) for Three.js visualization...
This is the same data shown in Napari viewer (Cell 8).
Data shape: (194, 688, 1363)
Data range: [0, 32737]
Data min: 0.0, max: 32737.0
Using full resolution (no downsampling)

✓ Data prepared for Three.js viewer!
✓ Files saved to: visualization_data
  - channel_19_napari_metadata.json
  - channel_19_napari_data.raw
✓ Data size: 173.49 MB

✓ Three.js HTML viewer created: visualization_data\threejs_volume_viewer_channel_19_napari.html
✓ Open this file in your browser to view the 3D volume
✓ This displays the same data as shown in Napari (Cell 8)
